In [2]:
print("Ok")

Ok


In [2]:
import os
from git import Repo
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv

In [4]:
import sys
print(sys.executable)

c:\Users\HP\.conda\envs\llmapp\python.exe


In [5]:
%pwd

'c:\\Auto-code-analyzer\\research'

In [6]:
!mkdir test_repo

In [11]:
try:
    repo.close()
except NameError:
    pass

In [1]:
import shutil, os, stat

def remove_readonly(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)

if os.path.exists("test_repo"):
    shutil.rmtree("test_repo", onerror=remove_readonly)

if os.path.exists("db"):
    shutil.rmtree("db", onerror=remove_readonly)

print("Cleanup done.")

Cleanup done.


In [3]:
repo_path = "test_repo/"
repo = Repo.clone_from("https://github.com/MalempatiGnapika/Auto-code-analyzer.git", to_path=repo_path)

In [4]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    suffixes=[".py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500),
)
documents = loader.load()
documents

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content=''),
 Document(metadata={'source': 'test_repo\\setup.py', 'language': <Language.PYTHON: 'python'>}, page_content=''),
 Document(metadata={'source': 'test_repo\\store_index.py', 'language': <Language.PYTHON: 'python'>}, page_content='import os\nfrom dotenv import load_dotenv\nfrom langchain_chroma import Chroma\n\nfrom src.helper import load_repo, text_splitter, load_embedding\n\nload_dotenv()\n\nCOHERE_API_KEY = os.environ.get("COHERE_API_KEY")\nos.environ["COHERE_API_KEY"] = COHERE_API_KEY\n\ndocuments = load_repo("repo/")\ntext_chunks = text_splitter(documents)\nembeddings = load_embedding()\n\nvectordb = Chroma.from_documents(\n    text_chunks,\n    embedding=embeddings,\n    persist_directory="./db",\n)'),
 Document(metadata={'source': 'test_repo\\template.py', 'language': <Language.PYTHON: 'python'>}, page_content='import os\nfrom pathlib import Path\nimport logging\n\nlogging

In [11]:
print(len(documents))
for doc in documents:
    print(doc.metadata["source"])

6
test_repo\app.py
test_repo\setup.py
test_repo\store_index.py
test_repo\template.py
test_repo\src\helper.py
test_repo\src\__init__.py


In [12]:
for doc in documents:
    print(doc.metadata["source"], "-", len(doc.page_content), "characters")

test_repo\app.py - 0 characters
test_repo\setup.py - 0 characters
test_repo\store_index.py - 464 characters
test_repo\template.py - 783 characters
test_repo\src\helper.py - 0 characters
test_repo\src\__init__.py - 0 characters


In [8]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=500, chunk_overlap=20
)
texts = documents_splitter.split_documents(documents)
len(texts)

3

In [13]:
print("Documents loaded:", len(documents))
print("Chunks created:", len(texts))
for t in texts:
    print(t.metadata["source"])

Documents loaded: 6
Chunks created: 3
test_repo\store_index.py
test_repo\template.py
test_repo\template.py


In [14]:
load_dotenv()
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [15]:
embeddings = CohereEmbeddings(model="embed-english-v3.0")

In [16]:
vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory="./db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [17]:
print(vectordb._collection.count())


3


In [18]:
llm = ChatCohere(model="command-a-03-2025", temperature=0.4)

In [19]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer it, just reformulate or return as-is."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 3})
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

In [20]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an assistant for question-answering tasks. Use the retrieved context to answer concisely.\n\n{context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [21]:
chat_history = []

In [22]:
question = "what is the repo_ingestion function doing?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 20 is greater than number of elements in index 3, updating n_results = 3


The provided code snippet does not explicitly define a `repo_ingestion` function. However, based on the context, it appears to be performing the following steps for repository ingestion:

1. **Loading Documents**: The `load_repo("repo/")` function is likely responsible for loading documents from a specified repository directory.
2. **Text Splitting**: The `text_splitter(documents)` function splits the loaded documents into smaller text chunks, which is a common preprocessing step for vectorization.
3. **Loading Embeddings**: The `load_embedding()` function loads a pre-trained embedding model, which is used to convert text chunks into vector representations.
4. **Creating a Vector Database**: The `Chroma.from_documents()` method creates a vector database (using Chroma) from the text chunks and their corresponding embeddings. The database is persisted to the `./db` directory.

These steps collectively form the repository ingestion process, where documents are loaded, processed, and store

In [23]:
import cohere

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
models = co.models.list()
for m in models.models:
    print(m.name, "-", m.endpoints)

c4ai-aya-expanse-32b - ['generate', 'chat']
c4ai-aya-vision-32b - ['chat']
cohere-transcribe-03-2026 - ['transcriptions']
command-a-03-2025 - ['chat']
command-a-plus-05-2026 - ['generate', 'chat']
command-a-reasoning-08-2025 - ['chat']
command-a-translate-08-2025 - ['chat']
command-a-vision-07-2025 - ['chat']
command-r-08-2024 - ['generate', 'chat', 'summarize']
command-r-plus-08-2024 - ['generate', 'chat', 'summarize']
command-r7b-12-2024 - ['generate', 'chat']
command-r7b-arabic-02-2025 - ['generate', 'chat']
embed-english-light-v3.0 - ['embed']
embed-english-light-v3.0-image - ['embed_image']
embed-english-v3.0 - ['embed']
embed-english-v3.0-image - ['embed_image']
embed-multilingual-light-v3.0 - ['embed']
embed-multilingual-light-v3.0-image - ['embed_image']
embed-multilingual-v3.0 - ['embed']
embed-multilingual-v3.0-image - ['embed_image']


In [24]:
question = "what is the text_splitter function doing?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 3, updating n_results = 3


The `text_splitter` function is responsible for dividing the loaded documents into smaller, manageable chunks of text. This is a crucial preprocessing step when working with large documents, as it helps in:

1. **Efficient Vectorization**: Smaller text chunks can be more efficiently converted into vector representations using embedding models.
2. **Improved Search and Retrieval**: Breaking down documents into chunks enables more granular search and retrieval, allowing for better matching of query terms with relevant portions of the text.
3. **Reduced Computational Overhead**: Processing smaller chunks of text reduces the computational resources required for tasks like embedding generation and database storage.

Although the exact implementation of the `text_splitter` function is not provided in the code snippet, it typically involves:

- **Splitting by a fixed size or overlap**: Dividing the text into chunks of a fixed size (e.g., 1000 characters) or with a specified overlap (e.g., 100

In [25]:
question = "what does the load_embedding function return?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 3, updating n_results = 3


The `load_embedding` function is likely responsible for loading a pre-trained embedding model that converts text into numerical vector representations. Based on the context, it probably returns an instance of an embedding model compatible with the `langchain_chroma` library and the Chroma vector database.

Here’s what the `load_embedding` function likely does and returns:

1. **Loads a Pre-trained Embedding Model**: It initializes an embedding model, such as those provided by libraries like **Cohere**, **OpenAI**, **Hugging Face**, or other embedding providers.
2. **Returns the Embedding Model Instance**: The function returns the loaded embedding model, which can then be used to generate vector embeddings for text chunks.

Given the use of `COHERE_API_KEY` in the environment variables, it’s likely that the `load_embedding` function is loading a **Cohere embedding model**. The return type would be an object or interface that can be passed to the `Chroma.from_documents` method for genera

In [26]:
question = "what does the /chatbot route in app.py do?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Number of requested results 20 is greater than number of elements in index 3, updating n_results = 3


The `/chatbot` route in `app.py` is not explicitly defined in the provided code snippet. However, based on typical web application structures, the `/chatbot` route would likely handle requests related to a chatbot functionality. Here’s what it might do in a standard setup:

1. **Endpoint for Chatbot Interaction**:  
   The `/chatbot` route would serve as an API endpoint where users can send messages (e.g., via POST requests) and receive responses from the chatbot.

2. **Processing User Input**:  
   It would take user input (e.g., a question or message), process it, and use the vector database (`vectordb`) or other models to generate a relevant response.

3. **Integration with Vector Database**:  
   If the chatbot relies on the vector database created earlier, the route might:
   - Query the `vectordb` using the user's input to retrieve relevant documents or context.
   - Use the retrieved information to formulate a response.

4. **Response Generation**:  
   The route would return th

In [27]:
question = "how does the app clear the cloned repository?"
result = rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
print(result["answer"])

Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised InternalServerError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'd3a0143c3a1583c663f078d7f2aacb75', 'date': 'Wed, 01 Jul 2026 21:08:29 GMT', 'x-envoy-upstream-service-time': '5396', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 500, body: {'id': '3558822a-e1e9-41e4-9d74-400b0ae6ab4f', 'message': 'internal server error, this has been reported to our developers. id d3a0143c3a1583c663f078d7f2aacb75'}.
Retrying langchain_cohere.embeddings.CohereEmbeddings

The provided code snippet does not explicitly show how the app clears or deletes a cloned repository. However, based on typical practices, here’s how such functionality might be implemented:

1. **Identify the Repository Directory**:  
   The app would need to know the path to the cloned repository directory. In the provided code, the repository is loaded from the `"repo/"` directory using `load_repo("repo/")`.

2. **Delete the Directory and Its Contents**:  
   To clear the cloned repository, the app would delete the directory and all its contents. This can be done using Python's `shutil` or `os` modules.

Here’s an example of how this might be implemented:

```python
import os
import shutil
import logging

def clear_repository(repo_path):
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)  # Deletes the directory and its contents
        logging.info(f"Repository at {repo_path} has been cleared.")
    else:
        logging.info(f"Repository at {repo_path} does not exi